# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install matplotlib numpy --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import numpy as np
import matplotlib.pyplot as plt

### 1. Scenario: Rocket Landing Process

SpaceX lands its returning rockets - specifically, the first stage of the Falcon 9 and Falcon Heavy rockets—using a sophisticated sequence of maneuvers, powerful engines, and precise guidance systems. SpaceX’s landing process involves precise control and the coordination of engines, fins, and legs to bring the rocket back safely. This reuse capability reduces the cost of launches, making space missions more sustainable and frequent.

After the rocket stage has completed its main mission and separated from the second stage, the first stage performs a boost-back burn to reverse its course and begin the return journey to the landing zone. This burn uses some of the engines to slow the stage down and adjust its trajectory, targeting either a drone ship in the ocean or a ground-based landing pad.

As the rocket re-enters the Earth's atmosphere, it performs a re-entry burn to reduce its speed and protect itself from the intense heat generated during atmospheric re-entry. This burn, using just a few of the engines, helps decrease velocity and reduces aerodynamic stress, preserving structural integrity.

SpaceX rockets are equipped with four grid fins, which deploy after re-entry to help control the rocket’s orientation and stability. These grid fins, which look like metal grids, steer the rocket by adjusting lift and drag. They allow fine control and help guide the rocket’s descent to stay on course toward the landing target.
   
As the rocket nears its landing site, it initiates a landing burn, firing one or more engines to slow down to a controlled descent rate. During this final burn, the rocket's velocity decreases dramatically, allowing it to reach near-zero speed just as it touches down.

Right before landing, the rocket deploys four landing legs. These legs are stowed along the body of the rocket during flight and deploy automatically to provide a stable base. The rocket touches down with its legs extended, ensuring a smooth and safe landing.

The Falcon 9 uses engines that can be restarted multiple times, which is crucial for performing the different burns required during descent. The rocket’s landing system is controlled by an onboard guidance computer, which uses data from sensors and GPS to precisely track the rocket’s position, speed, and orientation. During burns, the rocket adjusts its direction using thrust vectoring, where the engines pivot to control the trajectory. The rocket faces high g-forces as it rapidly decelerates for landing. The timing and power of the burns are finely tuned to minimize fuel use while maintaining control.

### 2. Python Example

In [ ]:
# --- Rocket Landing Simulation with PID Controller ---
GRAVITY = 9.81  # m/s^2, gravity of the planet
WIND_STD = 4  # Wind disturbance strength
DELTA_TIME = 0.1  # Time step for the simulation

# PID controller class
class PIDController:
    def __init__(self, kp, ki, kd):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.previous_error = 0
        self.integral = 0

    def update(self, error, dt):
        # Proportional, Integral, Derivative terms
        self.integral += error * dt
        derivative = (error - self.previous_error) / dt
        output = self.kp * error + self.ki * self.integral + self.kd * derivative
        self.previous_error = error
        return output

class Rocket:
    def __init__(self, position, velocity):
        self.position = np.array(position, dtype='float64')
        self.velocity = np.array(velocity, dtype='float64')
        self.mass = 1.0  # kg, mass of the rocket
        self.thrust = 0.0
        self.thrust_angle = 0.0

    def apply_thrust(self):
        thrust_x = self.thrust * np.sin(self.thrust_angle)
        thrust_y = self.thrust * np.cos(self.thrust_angle) - self.mass * GRAVITY
        return np.array([thrust_x, thrust_y]) / self.mass

    def update(self, force, dt):
        acceleration = force
        self.velocity += acceleration * dt
        self.position += self.velocity * dt
        
initial_position = [0, 100]  # Start at 100 meters above the ground
initial_velocity = [0, 0]    # Initial velocity
rocket = Rocket(initial_position, initial_velocity)

velocity_pid = PIDController(kp=1, ki=.7, kd=0)
#altitude_pid = PIDController(kp=1.0, ki=0.05, kd=0.3)
horizontal_pid = PIDController(kp=1.0, ki=0.05, kd=0.3)

positions = []
velocities = []
target_velocities = []
times = []
thrusts = []
thrust_angles = []

for t in np.arange(0, 500, DELTA_TIME):
    
    target_velocity = -0.2 - 0.1 * rocket.position[1];
    
    velocity_error = target_velocity - rocket.velocity[1];
    # altitude_error = -rocket.position[1]  # Target altitude is 0 (ground level)
    horizontal_error = -rocket.position[0]  # Target horizontal position is 0 (aligned with landing pad)

    thrust_magnitude = velocity_pid.update(velocity_error, DELTA_TIME)
    thrust_direction = horizontal_pid.update(horizontal_error, DELTA_TIME)

    rocket.thrust = max(0, min(thrust_magnitude, 20))  # Limit thrust power
    rocket.thrust_angle = np.arctan2(thrust_direction, 1)  # Directional angle adjustment

    wind_force = np.random.normal(0, WIND_STD)

    total_force = rocket.apply_thrust() + np.array([wind_force, 0])
    rocket.update(total_force, DELTA_TIME)

    positions.append(rocket.position.copy())
    velocities.append(rocket.velocity.copy())
    target_velocities.append(target_velocity.copy())
    thrusts.append(rocket.thrust)
    thrust_angles.append(rocket.thrust_angle.copy())
    times.append(t.copy())

    if rocket.position[1] <= 0:
        if abs(rocket.velocity[1]) < 0.5:
            print("Rocket touched ground and landed successfully.");
        else:
            print("Rocket touched ground but crashed :-( !");
        break


def plot_constellation(ax):
    ax.cla()
    ax.axis([None, None, None, None])
    ax.grid(True)
    ax.set_facecolor((.2, 0.2, 0.3))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
theLineWidth = 2
fig.set_facecolor((.6, 0.61, 0.8))

plot_constellation(axes[1])
plot_constellation(axes[0])    

velocities = np.array(velocities)
axes[0].plot(times, target_velocities, linewidth=theLineWidth, label="target vert veloc.")
axes[0].plot(times, velocities[:,1], linewidth=theLineWidth, label="vert velocity")
axes[0].plot(times, velocities[:,0], linewidth=theLineWidth, label="horiz velocity")
axes[0].plot(times, thrusts, linewidth=theLineWidth, label="thrust")
axes[0].plot(times, thrust_angles, linewidth=theLineWidth, label="thrust angle")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Controller In/Out")
axes[0].legend(fancybox=True, framealpha=1,loc="lower right", facecolor=(.6, 0.61, 0.8))

# Plot the trajectory
positions = np.array(positions)
axes[1].plot(positions[:, 0], positions[:, 1], linewidth=theLineWidth, label="Trajectory")
axes[1].plot(positions[len(positions)-1, 0], positions[len(positions)-1, 1], marker="X", markersize=10, linewidth=0, label="Landing Site")
axes[1].set_xlabel("Horizontal Position (m)")
axes[1].set_ylabel("Altitude (m)")
axes[1].legend(fancybox=True, framealpha=1,loc="upper left", facecolor=(.6, 0.61, 0.8))

plt.tight_layout()

<p align="center" width="100%">═════════════════════════════════<br>
 Stefan Mangold (<a href="mailto:stefan.mangold@inf.ethz.ch">stefan.mangold@inf.ethz.ch</a>)<br><img width="200px" height="auto" src="../Admin/eth_logo_kurz_pos.png"></p>